# 00: 環境と観測の確認

モデルを使わずにLift環境を動かし、画像・状態・行動と記録を確認します。

1. このノートブックを自分のDriveにコピーします。
2. ランタイム → ランタイムのタイプを変更 → **T4 GPU**を選びます。
3. 上から順に実行し、動画とログを確認します。最初は20ステップです。
4. 最後のセルで結果をDriveに保存し、ランタイムを接続解除して削除します。

この1冊に準備と実習コードを同梱しています。別冊の変数や実行結果、教材リポジトリのcloneは不要です。
ライブラリとモデルは実行時に取得します。Python 3.12／3.13向けで、初回準備には時間がかかります。

T4を要求する設定は入っていますが、空きや利用枠によって接続できない場合があります。
接続前の失敗では教材コードはまだ実行されていません。ノートブックから割り当てを強制することはできません。

確認済みの範囲は[検証状況](https://github.com/takumiecd/vla-workshop/blob/main/_dev/docs/validation.md)を参照してください。


In [ ]:
# T4とPythonを確認する（Colabへの接続完了後に実行）
import subprocess
import sys

print(sys.version)
if sys.version_info[:2] not in {(3, 12), (3, 13)}:
    raise RuntimeError("Python 3.12または3.13のランタイムを選んでください")
try:
    gpu_names = (
        subprocess.check_output(
            ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], text=True
        )
        .strip()
        .splitlines()
    )
except (FileNotFoundError, subprocess.CalledProcessError) as error:
    raise RuntimeError(
        "ランタイムのタイプをT4 GPUに変更し、接続してから実行してください"
    ) from error
print("割り当てられたGPU:", gpu_names)
if len(gpu_names) != 1 or "T4" not in gpu_names[0].upper():
    raise RuntimeError("ランタイム → ランタイムのタイプを変更 → T4 GPU を選んでください")

## 内蔵プログラムを準備する

このセルを実行すると、ipynbに同梱した実習コードをColab内に準備します。設定の変更は不要です。
教材リポジトリのcloneや、教材の.pyを別途ダウンロードする操作はありません。

ノートブックは共有Driveから開いても、GitHubの「Open in Colab」から開いても同じ内容です。
「ファイル → ドライブにコピーを保存」から自分用のコピーを作り、上から順に実行してください。


In [ ]:
# @title 内蔵プログラムを準備する（実行するだけ）
# 実習コードはこのipynbに同梱されています。教材リポジトリの取得は不要です。
import base64
import hashlib
import io
import json
import zipfile
from pathlib import Path

BUNDLE = (
    "..."
)
EXPECTED_SHA256 = "1cf99aa646555310d78939f854402b6c77056a3d4212f131ecce51de4177f142"
data = base64.b64decode(BUNDLE)
assert hashlib.sha256(data).hexdigest() == EXPECTED_SHA256, "内蔵コードの照合に失敗しました"
PROJECT = Path("/content/vla_workshop")
PROJECT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(data)) as archive:
    for name in archive.namelist():
        assert (PROJECT / name).resolve().is_relative_to(PROJECT.resolve()), name
    archive.extractall(PROJECT)
(PROJECT / "outputs").mkdir(exist_ok=True)
(PROJECT / "outputs/source_bundle.json").write_text(json.dumps({"sha256": EXPECTED_SHA256}) + "\n")
print("内蔵プログラムの準備ができました:", PROJECT)

## 共通環境をセットアップする

Pythonの版に対応する固定依存リストから、教材専用の環境を作ります。
Colab標準ライブラリの更新に影響されないよう、この環境のPythonから実習を実行します。
CUDA 12.8版のPyTorchと対応するtorchvisionを使います。

初回はCUDA関連パッケージも取得するため時間がかかります。進捗はセルと`outputs/setup.log`に表示します。
インストール後に`pip check`と主要ライブラリのimport、T4上のCUDA計算を確認してから次へ進みます。


In [ ]:
# @title 実習用の環境を準備する
subprocess.run(
    [sys.executable, str(PROJECT / "scripts/setup_colab.py"), "--project", str(PROJECT)], check=True
)
PYTHON = PROJECT / ".venv/bin/python"
print("実習用Python:", PYTHON)


def run_lesson(script, *, steps=20, episodes=1):
    result = subprocess.run(
        [
            str(PYTHON),
            str(PROJECT / script),
            "--config",
            str(PROJECT / "configs/lift.json"),
            "--max-steps",
            str(steps),
            "--episodes",
            str(episodes),
        ],
        cwd=PROJECT,
        text=True,
        capture_output=True,
    )
    print(result.stdout)
    if result.stderr:
        print(result.stderr[-6000:])
    result.check_returncode()
    line = next(line for line in result.stdout.splitlines() if line.startswith("Results: "))
    return Path(line.removeprefix("Results: "))

## 00: 環境と観測を確認する

標準Lift環境にPandaを置きます。画像・状態・制御・動画保存を確認します。
この段階は学習モデルを使わず、手先の移動指令をゼロ、グリッパーを開く指令にします。

In [ ]:
inspect_result = run_lesson("lessons/00_environment/run.py")

In [ ]:
from IPython.display import Image, Video, display

display(Image(filename=str(inspect_result / "episode_000/agentview.png")))
display(Video(str(inspect_result / "episode_000/video.mp4"), embed=True))
print((inspect_result / "summary.json").read_text())
print((inspect_result / "episode_000/environment.json").read_text())

### 確認課題

- 画像にはPandaと立方体が写っていますか。
- 8次元の状態と7次元の行動について、各要素の意味を説明できますか。
- 20 Hzという設定と、このセルの実行時間の違いを説明できますか。

詳しい仕様は教材内の`docs/io_contract.md`、読むコードは`src/vla_workshop/envs/robosuite_env.py`です。

## 結果をDriveへ保存して終了する

動画・ログ・実行条件をまとめて、自分のDriveに保存します。Google Driveへの接続承認が必要です。
`RESULTS_DIR`を変更すれば、共有フォルダ内の自分の結果フォルダにも保存できます。
共有フォルダを使う場合は、自分のMyDriveにそのフォルダのショートカットを追加してパスを指定してください。

保存後、ランタイムの「接続解除して削除」を実行します。


In [ ]:
# @title 結果をDriveへ保存する
import shutil
from datetime import UTC, datetime
from uuid import uuid4

from google.colab import drive

RESULTS_DIR = "/content/drive/MyDrive/vla_Workshop/results"  # @param {type:"string"}
drive.mount("/content/drive")
destination = Path(RESULTS_DIR)
destination.mkdir(parents=True, exist_ok=True)
timestamp = datetime.now(UTC).strftime("%Y%m%dT%H%M%SZ")
name = f"00_environment_results_{timestamp}_{uuid4().hex[:8]}"
archive = shutil.make_archive("/content/" + name, "zip", PROJECT / "outputs")
saved = shutil.copy2(archive, destination / (name + ".zip"))
print("Driveに保存しました:", saved)